# Day 19 — Vision-Language Modeling for Diagnosis

**Agri-AI Bootcamp** · Phase 5 of 5: *Transformers & Ag-Agents* · Days 17–20

> Load Salesforce/blip-image-captioning-base and generate text for plant-disease or farm images.

**Runtime:** `Runtime -> Change runtime type -> T4 GPU` before you start.

**How to use this notebook:** run the cells top to bottom. The *Worked solution*
section is complete and runnable; the *Your turn* section at the end is where you
experiment. Datasets that need a Kaggle/Roboflow key are simulated here with the
same columns and shapes, and the real download command is shown in a comment —
swap it in when you have the credentials. Example numbers are illustrative.

## 1. Explained

A vision-language model encodes the image and passes those features to a language decoder that generates text. BLIP-base can caption an image or continue a text prompt. Visual question answering asks a direct question about the picture.

### 🌾 Agriculture example

Prompt “a close up photo of a plant leaf showing symptoms of” and read the continuation; use it to draft notes for an agronomist to verify.

> **Watch out:** Generated text can sound confident and be wrong; show it beside the image with a human check.

## 2. Setup

In [ ]:
!pip -q install transformers

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

## 3. Data

In [ ]:
# ---- Leaf photos (swap in real PlantVillage images when you have them) ------
rng = np.random.default_rng(1)
def leaf(kind, size=224):
    a = rng.uniform(0, .05, (size, size, 3)) + [.16, .52, .20]
    yy, xx = np.ogrid[:size, :size]
    a[((xx-112)/95)**2 + ((yy-112)/70)**2 > 1] = [.92, .92, .90]     # white background
    if kind != "healthy":
        for _ in range(7):
            cx, cy = rng.integers(55, size-55, 2); r = rng.integers(10, 20)
            m = (xx-cx)**2 + (yy-cy)**2 <= r**2
            a[m] = [.48, .30, .12] if kind == "spots" else [.85, .80, .25]
    return Image.fromarray((a.clip(0,1)*255).astype(np.uint8))

leaves = {k: leaf(k) for k in ["healthy", "spots", "yellowing"]}
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, (k, im) in zip(ax, leaves.items()):
    a.imshow(im); a.set_title(k, fontsize=9); a.axis("off")
plt.show()

## 4. Starter code from the course

Reference only — the shortest form of today's idea. It assumes variables that the
next section defines properly, so read it, don't run it. Section 5 is the runnable
version.

```python
from transformers import BlipProcessor, BlipForConditionalGeneration
n="Salesforce/blip-image-captioning-base"
pr=BlipProcessor.from_pretrained(n); m=BlipForConditionalGeneration.from_pretrained(n)
x=pr(img,"A close up photo of a plant leaf showing symptoms of ",return_tensors="pt")
print(pr.decode(m.generate(**x,max_new_tokens=30)[0],skip_special_tokens=True))
```

## 5. Worked solution

In [ ]:
name = "Salesforce/blip-image-captioning-base"
proc  = BlipProcessor.from_pretrained(name)
model = BlipForConditionalGeneration.from_pretrained(name).to(dev).eval()
print("params:", f"{sum(p.numel() for p in model.parameters())/1e6:.0f}M")

def caption(img, prompt=None, **kw):
    x = proc(img, prompt, return_tensors="pt").to(dev) if prompt else \
        proc(img, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = model.generate(**x, max_new_tokens=30, **kw)
    return proc.decode(out[0], skip_special_tokens=True)

print("\n--- unconditional captioning ---")
for k, im in leaves.items():
    print(f"{k:<10} -> {caption(im)!r}")

print("\n--- prompted continuation (steer the decoder) ---")
PROMPT = "a close up photo of a plant leaf showing symptoms of"
for k, im in leaves.items():
    print(f"{k:<10} -> {caption(im, PROMPT)!r}")

# ---- Decoding strategy changes the text, not the image ----------------------
print("\n--- same image, three decoding settings ---")
im = leaves["spots"]
print("greedy      :", caption(im, PROMPT))
print("beam search :", caption(im, PROMPT, num_beams=5))
print("sampling    :", caption(im, PROMPT, do_sample=True, top_p=0.9, temperature=0.9))

# ---- Visual question answering ----------------------------------------------
# BLIP-base captioning can be prompted question-style; for proper VQA load
# "Salesforce/blip-vqa-base" instead.
print("\n--- question-style prompts ---")
for q in ["Question: are there spots on this leaf? Answer:",
          "Question: what colour is this leaf? Answer:"]:
    print(f"{q}\n   -> {caption(im, q)!r}")

# ---- A draft note for the agronomist, with the caveat attached --------------
def draft_note(img, field_id):
    text = caption(img, PROMPT)
    return (f"Field {field_id} -- AI DRAFT, NOT A DIAGNOSIS\n"
            f"  model observation: {text}\n"
            f"  action: forward the photo to an agronomist for confirmation\n"
            f"  reviewed by: ______________   date: __________")

print("\n" + draft_note(leaves["spots"], "B-12"))

print("""
WHY THE CAVEAT MATTERS
A vision-language model produces fluent text whether or not it recognised
anything. It has no notion of being unsure. In an advisory product:
  * always show the image next to the generated text
  * keep a human in the loop before any spraying or treatment advice
  * log the model version, so a wrong note can be traced
""")

## 6. Your turn

1. Caption the same leaf 5 times with sampling on. How stable is the text? That is your reliability estimate.
2. Load `Salesforce/blip-vqa-base` and ask direct questions instead of prompting a caption.
3. Pair BLIP's caption with your Day 11 classifier's label and flag disagreements for review.

In [ ]:
# TODO 1: stability under sampling
for i in range(5):
    print(i, caption(leaves["spots"], PROMPT, do_sample=True, top_p=.9, temperature=1.0))

# TODO 2: proper VQA
# from transformers import BlipForQuestionAnswering
# vqa = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to(dev)

# TODO 3: agreement check between the classifier label and the caption

## 7. Dataset and references

**Dataset:** [Kaggle: PlantVillage leaf images](https://www.kaggle.com/datasets/emmarex/plantdisease)

**References**

- [BLIP model card](https://huggingface.co/Salesforce/blip-image-captioning-base)
- [Hugging Face Transformers docs](https://huggingface.co/docs/transformers)

**Course material:** Hugging Face Multimodal Models guide

---
**Next:** Day 20 — Capstone: Smart Farming Assistant